# Geometry, archetype, inspection

Choose geometry explicitly, then apply a catalogue or authored Data definition.
This offline example uses no database, weather file or simulation engine.
Missing systems and site conditions remain visible.

In [ ]:
from epsm_core import FacadeSpec, Footprint, GlazingSpec
from epsm_data import Catalogue, select_operating_profile

from epsm import Building

building = Building.from_footprint(
    Footprint.rectangle(12, 8), storeys=2, storey_height_m=2.8
)
# Explicit illustrative geometry choices; the archetype does not select WWR.
building.facade = FacadeSpec(
    default_wwr=0.2,
    glazing=GlazingSpec(u_value_w_m2k=1.4, solar_heat_gain_coefficient=0.55),
)
geometry = building.geometry
openings = [(e.identifier, e.area_m2) for e in building.elements]

The researcher supplies the load-reference area. Applying Atemp-based inputs
asserts that this explicit area represents Atemp for this example; it is an
assumption rather than a measured building fact.

In [ ]:
building.load_reference_area_m2 = 180
selected = Catalogue.load().get("mf_se02", state="existing")
# Profile selection is explicit here; catalogue associations can also supply it.
archetype = selected.with_changes(
    operating_profile=select_operating_profile("multi_family")
)
report = building.apply_archetype(archetype, calendar={"calendar_year": 2023})
assert building.geometry == geometry
assert [(e.identifier, e.area_m2) for e in building.elements] == openings
assert building.hvac is None  # No installed topology or heating system was chosen.
report.summary()

Catalogue and custom definitions share the same immutable authoring model.
The custom edit below changes construction performance without recreating any
opening. `fill_missing` would preserve already assigned windows instead.

In [ ]:
from epsm_data import Archetype, ConstructionSet
from epsm_data.domain import BuildingElement as Role

window = archetype.construction_set.elements[Role.WINDOW].with_changes(u_value=0.9)
authored = Archetype(
    "example:window-study",
    "Custom window definition",
    construction_set=ConstructionSet(elements={Role.WINDOW: window}),
    source_references=("example:researcher-selected-window-performance",),
)
custom_report = building.apply_archetype(authored)
assert [(e.identifier, e.area_m2) for e in building.elements] == openings
assert len(building.archetype_applications) == 2
assert (
    Building.from_dict(building.to_dict()).archetype_applications[0].to_dict()
    == report.to_dict()
)
custom_report.summary()

Inspect `report.requirements` before simulation. Choose installed HVAC,
reference-basis conversions and site ground inputs explicitly. The application
report does not establish scientific calibration or simulation readiness.